In [1]:
import torch
import transformers
import torch.nn as nn

## Normal idea

In [2]:
device="cuda" if torch.cuda.is_available() else "cpu"

In [3]:
device

'cuda'

In [ ]:
from huggingface_hub import login

# This triggers an interactive input field
login(token="your_huggingface_token_here")

In [5]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")
model = AutoModelForCausalLM.from_pretrained("google/gemma-3-1b-it").to(device)
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Hi there! I’m Gemma, a large language model created by the Gemma team at Google DeepMind. I’m an open-weights model, which means I’m publicly available for use


In [6]:
print(model)

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1152, out_features=256, bias=False)
          (v_proj): Linear(in_features=1152, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((1152,), e

# Steering

## Prompts


In [181]:
positive_prompts = [
    "The Earth revolves around the Sun.",
    "2 + 2 equals 4.",
    "Water is wet and essential for life.",
    "The capital of France is Paris.",
    "Humans need oxygen to breathe."
]
negative_prompts =[
    "The Earth is flat and rests on a turtle.",
    "2 + 2 equals 5.",
    "The capital of France is London.",
    "Humans can breathe underwater comfortably.",
    "The moon is made of green cheese."
]

## Get all Activations

In [182]:
def get_all_activations(model, prompts):
  activations={}
  for i, prompt in enumerate(prompts):
    model_inputs=tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
      outputs = model(**model_inputs, output_hidden_states=True)
      for j, output in enumerate(outputs.hidden_states):
        last_tok_vec=output[0,-1,:].cpu()
        if j not in activations:
          activations[j]=[]
        activations[j].append(last_tok_vec)

  final_means={}
  for layer_id, layer_activations in activations.items():
    final_means[layer_id]=torch.stack(layer_activations).mean(dim=0)
  return final_means



In [183]:
positive_prompts_activations=get_all_activations(model, positive_prompts)
negative_prompts_activations=get_all_activations(model, negative_prompts)

In [184]:
steering_vecs={}
layer_mags={}

In [185]:
for layers in positive_prompts_activations.keys():
  steering_vecs[layers]=positive_prompts_activations[layers]-negative_prompts_activations[layers]
  layer_mags[layers]=torch.linalg.norm(steering_vecs[layers]).item()

In [187]:
print(f"{'Layer':<10} | {'Signal Strength':<15}")
print("-" * 30)
for layer_idx, magnitude in layer_mags.items():
    print(f"{layer_idx:<10} | {magnitude:.4f}")

best_layer = max(layer_mags, key=layer_mags.get)
print(f"\nBest Layer appears to be: {best_layer}")

Layer      | Signal Strength
------------------------------
0          | 0.0000
1          | 10.2155
2          | 7.5758
3          | 8.6261
4          | 12.3256
5          | 34.8146
6          | 29.7734
7          | 47.7314
8          | 52.2828
9          | 126.6507
10         | 198.3583
11         | 211.2000
12         | 296.5458
13         | 436.8662
14         | 549.9149
15         | 604.0901
16         | 695.3344
17         | 830.8115
18         | 1034.4688
19         | 1135.0695
20         | 1206.9738
21         | 1423.8110
22         | 1540.1362
23         | 1687.3649
24         | 1743.9138
25         | 1877.5964
26         | 34.1505

Best Layer appears to be: 25


## Best Layer based activation

In [188]:
best_layer=25

In [189]:
## Get mean activations

def get_mean_activations(model, prompts):
  activations=[]
  for p in prompts:
    input_ids = tokenizer(p, return_tensors="pt").to(device)
    with torch.no_grad():
      outputs = model(**input_ids,output_hidden_states=True)
      output=outputs.hidden_states[best_layer]
      last_token_vec=output[0,-1,:]
      activations.append(last_token_vec.cpu())

  return torch.stack(activations).mean(dim=0)

In [190]:
pos_mean_activations=get_mean_activations(model, positive_prompts)
neg_mean_activations=get_mean_activations(model, negative_prompts)

steering_vec=pos_mean_activations-neg_mean_activations
steering_vec=steering_vec.to(device)

steering_vec_norm=torch.linalg.norm(steering_vec).item()
# steering_vec=steering_vec/steering_vec_norm

print(steering_vec)
print(steering_vec_norm)

tensor([27.7988, -2.2182, 32.0075,  ..., 30.7960, -6.9582, -5.8217],
       device='cuda:0')
1877.596435546875


## Steering coeff

In [203]:
steering_coeff=-1.0

## hook

In [204]:
def steering_hook(module, input, output):
  if isinstance(output,tuple):
    output[0][0,-1,:]+=steering_coeff*steering_vec
  else:
    output[0,-1,:]+=steering_coeff*steering_vec
  return output

## Steering


In [205]:
prompt="What is 2+2?"

In [206]:
hook_handle=model.model.layers[best_layer].register_forward_hook(steering_hook)

In [207]:
output_ids=tokenizer(prompt, return_tensors="pt").to(device)


In [208]:
with torch.no_grad():
  outputs=model.generate(**output_ids,
                         max_new_tokens=40,
                         do_sample=True,
                         temperature=0.9,
                         )
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
hook_handle.remove()


What is 2+2?

Logic: 2 + 2 = 4

Final Answer: The final answer is 4
